Stayman 2H strategy analysis after `1NT - 2C - 2D`.

This notebook compares two competing meanings for the responder's `2H` rebid:

- **Strategy A**: `2H` is weak crawling, showing equal-length majors. The fallback invitational continuations are `3H`, `3N`, or `4H` depending on opener strength and 3 hearts.
- **Strategy B**: `2H` is invitational, showing 5 hearts and 4 spades with 8 HCP, while weak 4-4 major responder hands are instead left in `1NT` and may choose `2H` or `2S` only when opener prefers one major.

The analysis covers:

1. Relative frequency of weak 4-4 major and invitational 5H-4S responder patterns among all 15-17 balanced `1NT` openers without a 4-card major.
2. A head-to-head IMP comparison between the two strategies on the combined sample of both responder hand types.
3. Breakdowns for the weak 4-4 major sample and the invitational 5H-4S sample.

In [1]:
// Because of class loading reasons, this has to be loaded before or at the same cell as ReKtDeal
// for the `toFormattedFrame` and `toDataFrame` methods to work.
%use dataframe

@file:DependsOn("com.github.phisgr:rektdeal:0.3.0")

import com.github.phisgr.dds.*
import com.github.phisgr.dds.Deal as DdsDeal
import com.github.phisgr.rektdeal.*

println("Launching with $threadCount threads.")

Launching with 4 threads.


In [2]:
val bal1NT = Shape.balanced
val openerNT = SmartStack(bal1NT, Evaluator.hcp, 15..17)

val contract1NT = Contract("1N")
val contract2H = Contract("2H")
val contract2S = Contract("2S")
val contract3H = Contract("3H")
val contract3N = Contract("3N")
val contract4H = Contract("4H")

fun Deal.no4CardMajorOpener() = Shape.balanced(north) && north.hearts.size < 4 && north.spades.size < 4
fun Deal.isWeak44Majors() = south.hearts.size == 4 && south.spades.size == 4 && south.hcp <= 7
fun Deal.isInv54Hearts() = south.hearts.size == 5 && south.spades.size == 4 && south.hcp == 8

fun Deal.openerLongerMajor() = if (north.hearts.size >= north.spades.size) H else S
fun Deal.openerHas3Hearts() = north.hearts.size == 3
fun Deal.openerStrong() = north.hcp >= 17

fun score(contract: Contract, deal: Deal, vulnerable: Boolean): Int {
    val tricks = deal.ddTricks(contract.strain, SOUTH)
    return contract.score(tricks, vulnerable)
}

fun scoreStrategy(deal: Deal, vulnerable: Boolean, strategy: (Deal) -> Contract): Int =
    score(strategy(deal), deal, vulnerable)

fun strategyCrawling(deal: Deal): Contract = when {
    deal.isWeak44Majors() -> when (deal.openerLongerMajor()) {
        H -> contract2H
        S -> contract2S
        else -> contract2H
    }
    deal.isInv54Hearts() -> when {
        !deal.openerStrong() -> contract3H
        deal.openerHas3Hearts() -> contract4H
        else -> contract3N
    }
    else -> throw IllegalStateException("Unexpected hand for crawling strategy")
}

fun strategyInvitational(deal: Deal): Contract = when {
    deal.isWeak44Majors() -> contract1NT
    deal.isInv54Hearts() -> contract2H
    else -> throw IllegalStateException("Unexpected hand for invitational strategy")
}

In [3]:
fun sampleResponderPatterns(count: Int = 30_000): Map<String, Long> {
    val states = multiThread(
        count = count,
        dealer = { Dealer(N = openerNT) },
        state = { LongArray(3) },
        accept = { deal -> deal.no4CardMajorOpener() },
        action = { _, deal, state ->
            when {
                deal.isWeak44Majors() -> state[0]++
                deal.isInv54Hearts() -> state[1]++
                else -> state[2]++
            }
        }
    )

    return mapOf(
        "weak44Majors" to states.sumOf { it[0] },
        "inv54Hearts" to states.sumOf { it[1] },
        "other" to states.sumOf { it[2] }
    )
}

fun headToHeadPayOff(
    count: Int = 25_000,
    vulnerable: Boolean = false,
): PayOff<String> {
    val states = multiThread(
        count = count,
        dealer = { Dealer(N = openerNT) },
        state = { PayOff(listOf("crawling", "invitational"), PayOff.impFromScores) },
        accept = { deal ->
            deal.no4CardMajorOpener() && (deal.isWeak44Majors() || deal.isInv54Hearts())
        },
        action = { _, deal, state ->
            state.addData(mapOf(
                "crawling" to scoreStrategy(deal, vulnerable, ::strategyCrawling),
                "invitational" to scoreStrategy(deal, vulnerable, ::strategyInvitational),
            ))
        }
    )

    return states.reduce { acc, payOff -> acc.combine(payOff) }
}

fun samplePayOff(
    accept: (Deal) -> Boolean,
    count: Int = 20_000,
    vulnerable: Boolean = false,
): PayOff<String> {
    val states = multiThread(
        count = count,
        dealer = { Dealer(N = openerNT) },
        state = { PayOff(listOf("crawling", "invitational"), PayOff.impFromScores) },
        accept = accept,
        action = { _, deal, state ->
            state.addData(mapOf(
                "crawling" to scoreStrategy(deal, vulnerable, ::strategyCrawling),
                "invitational" to scoreStrategy(deal, vulnerable, ::strategyInvitational),
            ))
        }
    )
    return states.reduce { acc, payOff -> acc.combine(payOff) }
}

In [4]:
val distribution = sampleResponderPatterns(count = 40_000)
val total = distribution.values.sum()
println("Distribution among $total 1NT opener deals without a 4-card major:")
for ((name, value) in distribution) {
    println("- $name: $value (${100.0 * value / total}%)")
}

Distribution among 40000 1NT opener deals without a 4-card major:
- weak44Majors: 1317 (3.2925%)
- inv54Hearts: 107 (0.2675%)
- other: 38576 (96.44%)


In [5]:
val combinedPayOff = headToHeadPayOff(count = 1000, vulnerable = false)
println("\nCombined head-to-head IMP comparison (weak 4-4 majors + 5H-4S invitational):")
combinedPayOff.report()


Combined head-to-head IMP comparison (weak 4-4 majors + 5H-4S invitational):
        crawlin invitat 
crawlin         -0.39   
                (0.08)  
invitat +0.39           
        (0.08)          


In [ ]:
val weak44PayOff = samplePayOff(
    accept = { deal -> deal.no4CardMajorOpener() && deal.isWeak44Majors() },
    count = 20_000,
    vulnerable = false,
)

println("\nWeak 4-4 majors, non-vulnerable:")
weak44PayOff.report()

val inv54PayOff = samplePayOff(
    accept = { deal -> deal.no4CardMajorOpener() && deal.isInv54Hearts() },
    count = 20_000,
    vulnerable = false,
)

println("\nInvitational 5H-4S, non-vulnerable:")
inv54PayOff.report()

### Interpretation

- The win/loss comparison is now head-to-head: every accepted deal sees both strategies applied and the IMP difference is measured directly.
- Weak 4-4 major responder hands compare `1NT` against `2H` or `2S` depending on which major opener is longer with.
- Invitational 5H-4S responder hands compare the crawling continuations (`3H`, `4H`, or `3N`) against the standard `2H` invitational treatment.

You can extend this notebook to include vulnerability, to compare exact 2H/2S selection rules, or to log the actual contract choices for each strategy.